# Seattle Building Permits — Exploratory Data Analysis

This notebook pulls the Seattle Building Permits dataset from the Seattle Open Data portal and explores it with a focus on accessory dwelling units (ADUs). The goal is to understand what people are actually building in Seattle backyards — which is often clearer than reading the municipal code.

**Dataset:** [Seattle Building Permits](https://data.seattle.gov/Permitting/Building-Permits/76t5-zqzr)  
**Source:** Seattle Open Data portal (Socrata API)  
**Output:** `data/raw/seattle_building_permits.csv`

## 1. Setup

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RAW_DATA_PATH = Path("../data/raw/seattle_building_permits.csv")
SOCRATA_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
DATASET_ID = "76t5-zqzr"
BASE_URL = f"https://data.seattle.gov/resource/{DATASET_ID}.json"

## 2. Download dataset

Pull the full dataset from the Socrata API using pagination (500 rows per request). Skips the download if the file already exists locally.

In [ ]:
def fetch_all_records(base_url: str, token: str | None, limit: int = 50000) -> list[dict]:
    """Page through the Socrata API and return all records."""
    headers = {"X-App-Token": token} if token else {}
    records = []
    offset = 0

    while True:
        resp = requests.get(
            base_url,
            headers=headers,
            params={"$limit": limit, "$offset": offset},
        )
        resp.raise_for_status()
        batch = resp.json()
        records.extend(batch)
        print(f"  fetched {len(records):,} records...", end="\r")
        if len(batch) < limit:
            break
        offset += limit

    print(f"\nDone — {len(records):,} total records")
    return records


if RAW_DATA_PATH.exists():
    print(f"File already exists at {RAW_DATA_PATH} — skipping download")
else:
    print("Fetching Seattle Building Permits...")
    records = fetch_all_records(BASE_URL, SOCRATA_TOKEN)
    df_raw = pd.DataFrame(records)
    RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(RAW_DATA_PATH, index=False)
    print(f"Saved to {RAW_DATA_PATH}")

In [ ]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

## 3. Schema inspection

Profile every column: type, null rate, unique value count, and sample values. Anything surprising gets flagged — these observations feed directly into the dbt staging model and test design.

In [ ]:
# Column-level profile: type, null rate, unique count, sample values
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "unique_count": df.nunique(),
    "sample_values": [df[c].dropna().unique()[:3].tolist() for c in df.columns],
})
profile

In [ ]:
# Columns with high null rates (>50%) — flag as unreliable for dbt tests
high_null = profile[profile["null_pct"] > 50].sort_values("null_pct", ascending=False)
print(f"{len(high_null)} columns with >50% nulls:")
high_null[["dtype", "null_pct"]]

In [ ]:
# Date columns — check parsing and range
date_cols = [c for c in df.columns if "date" in c.lower()]
for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    valid = parsed.dropna()
    print(f"{col}: {valid.min().date()} → {valid.max().date()} ({parsed.isnull().sum():,} unparseable)")

In [ ]:
# Permit type distribution — what categories exist?
permit_type_col = next((c for c in df.columns if "permittype" in c.lower() or "permit_type" in c.lower()), None)
print(f"Using column: {permit_type_col}")
df[permit_type_col].value_counts().head(20)